# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR² dataset on second primary colorectal cancer (CRC) in cancer survivors, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library for data access and processing.

### Dataset Source
The dataset source is defined using a Croissant schema file hosted at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load FAIR² dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review the available record sets, their `@id` values, and the fields they contain.

All record sets and fields are referenced by their Croissant `@id` property for transparency and reproducibility.

In [ ]:
# List all record set IDs and their fields

record_sets = dataset.record_sets()
print("Available record sets:")
for record_set in record_sets:
    print(f" - @id: {record_set.id}")
    print(f"   name: {record_set.name}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("   Fields:")
        for field in record_set.fields:
            print(f"    - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")
    elif hasattr(record_set, 'columns') and record_set.columns:
        print("   Columns:")
        for column in record_set.columns:
            print(f"    - @id: {column.id}, name: {column.name}, dataType: {getattr(column, 'data_type', None)}")
    else:
        print("   No fields/columns found.")
    print()

## 3. Data Extraction

Load all records from the main record set (typically the clinical data table) into a DataFrame for analysis.

*All references use Croissant `@id` identifiers for full traceability.*

In [ ]:
# Collect IDs for available recordsets
record_set_ids = [rset.id for rset in dataset.record_sets()]
print("All record set @id values:")
for rec_id in record_set_ids:
    print(f" - {rec_id}")

# For this dataset, select the main data recordset (often only one, confirm ID from the previous cell)
# In FAIR^2, typically biomedical tables use @id like 'https://api.app.sen.science/frontiers/7862866/clinical-table' or similar, adapt below:

chosen_record_set_id = record_set_ids[0]  # If only one recordset, else set explicitly

# Load all records from the chosen record set into a DataFrame
records = list(dataset.records(record_set=chosen_record_set_id))
df = pd.DataFrame(records)

print(f"Data columns in record set {chosen_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Explore, filter, and transform the dataset. All columns and fields are referenced by their Croissant `@id`.

We'll demonstrate:
- Filtering based on age (if available), removing outliers, and normalizing age.
- Grouping by sex (if available) or another categorical field.

In [ ]:
# Choose a numeric field by @id, e.g., 'age' column (e.g., 'http://id/age') -
# adapt according to the printed column names in the previous cell.

numeric_field_id = None
# Try to find a field related to age (use lower-case matching)
for col in df.columns:
    if "age" in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No 'age' field found; please adapt numeric_field_id manually.")
else:
    print(f"Selected numeric field: {numeric_field_id}")

# Filter records to those with age > 50 (adjust threshold as fits clinical logic)
if numeric_field_id is not None:
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, field_norm]].head())

    # Try grouping by sex or available categorical variable
    group_field_id = None
    for col in filtered_df.columns:
        if "sex" in col.lower() or "gender" in col.lower():
            group_field_id = col
            break

    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped (mean) by {group_field_id}:")
        print(grouped_df[[numeric_field_id, field_norm]])

## 5. Visualization

Visualize data distributions and relationships. We'll plot age distribution and group means (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of age (if field exists)
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='steelblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group field (e.g. sex, if present)
if numeric_field_id and group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id], palette='pastel')
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

- FAIR² demonstrates structured sharing of rich clinical CRC survivor data via the Croissant format.
- Using `mlcroissant`, you can:
    - Inspect available record sets and fields by `@id` for clear provenance
    - Load clinical tables into pandas DataFrames and apply standard EDA
    - Perform field-based filtering, normalization, and grouping for hypothesis generation
- Croissant's use of globally unique identifiers facilitates reproducible and machine-actionable biomedical analyses.